# Sampling

We want to collect reviews wrote by active users, meaning :
- Users with more than 20 reviews
- A sample of 50,000 users among them 
- For active each user, collect all his reviews

### First things first ###
Convert the JSONL to Parquet first. Every subsequent read will be 10-50x faster:

In [8]:
import polars as pl
import duckdb

# With Polars:
pl.scan_ndjson("data/Books.jsonl").sink_parquet("data/Books.parquet")

# With DuckDB:
duckdb.sql("""
    COPY (SELECT * FROM read_json_auto('data/Books.jsonl', format='newline_delimited'))
    TO 'data/Books.parquet' (FORMAT PARQUET, ROW_GROUP_SIZE 1000000)
""")

### First Sampling iteration Streaming Python ###

The simplest and most memory-efficient. Uses almost zero RAM beyond the dictionaries.


In [9]:
import json
import random
from collections import Counter
from datetime import datetime

DATA_PATH = "data/Books.jsonl"
OUTPUT_PATH = "sample-streaming-python/sampled_reviews.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Pass 1: Count reviews per user ──────────────────────────────
print("Pass 1: Counting reviews per user...")
user_counts = Counter()

with open(DATA_PATH, "r") as f:
    for i, line in enumerate(f):
        record = json.loads(line)
        user_counts[record["user_id"]] += 1
        if i % 5_000_000 == 0:
            print(f"  Processed {i:,} lines...")

print(f"Total unique users: {len(user_counts):,}")

# ── Filter active users (>= 20 reviews) ────────────────────────
active_users = [uid for uid, count in user_counts.items() if count >= MIN_REVIEWS]
print(f"Active users (>= {MIN_REVIEWS} reviews): {len(active_users):,}")

# ── Sample 50,000 users ────────────────────────────────────────
random.seed(SEED)
sampled_users = set(random.sample(active_users, min(NUM_USERS, len(active_users))))
print(f"Sampled users: {len(sampled_users):,}")

del user_counts, active_users  # free memory

# ── Pass 2: Extract reviews for sampled users ──────────────────
print("Pass 2: Extracting reviews for sampled users...")
total_extracted = 0

with open(DATA_PATH, "r") as fin, open(OUTPUT_PATH, "w") as fout:
    for i, line in enumerate(fin):
        record = json.loads(line)
        if record["user_id"] in sampled_users:
            fout.write(line)
            total_extracted += 1
        if i % 5_000_000 == 0:
            print(f"  Processed {i:,} lines, extracted {total_extracted:,}")

print(f"Total extracted reviews: {total_extracted:,}")
    


Pass 1: Counting reviews per user...
  Processed 0 lines...
  Processed 5,000,000 lines...
  Processed 10,000,000 lines...
  Processed 15,000,000 lines...
  Processed 20,000,000 lines...
  Processed 25,000,000 lines...
Total unique users: 10,297,355
Active users (>= 20 reviews): 137,305
Sampled users: 50,000
Pass 2: Extracting reviews for sampled users...
  Processed 0 lines, extracted 0
  Processed 5,000,000 lines, extracted 745,349
  Processed 10,000,000 lines, extracted 1,323,541
  Processed 15,000,000 lines, extracted 1,852,537
  Processed 20,000,000 lines, extracted 2,204,841
  Processed 25,000,000 lines, extracted 2,428,722
Total extracted reviews: 2,442,098


### Second Sampling iteration Using Pandas with Chunked Reading ###

In [10]:
import pandas as pd
import random
from collections import Counter

DATA_PATH = "data/Books.jsonl"
CHUNK_SIZE = 500_000  # rows per chunk
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Pass 1: Count reviews per user (chunked) ───────────────────
print("Pass 1: Counting reviews per user...")
user_counts = Counter()

reader = pd.read_json(DATA_PATH, lines=True, chunksize=CHUNK_SIZE,
                       dtype={"user_id": str, "parent_asin": str, "rating": float})

for i, chunk in enumerate(reader):
    counts = chunk["user_id"].value_counts()
    user_counts.update(counts.to_dict())
    print(f"  Chunk {i}: {len(chunk):,} rows processed")

# ── Filter & sample active users ───────────────────────────────
active_users = [uid for uid, c in user_counts.items() if c >= MIN_REVIEWS]
print(f"Active users: {len(active_users):,}")

random.seed(SEED)
sampled_users = set(random.sample(active_users, min(NUM_USERS, len(active_users))))
del user_counts, active_users

# ── Pass 2: Collect reviews for sampled users ──────────────────
print("Pass 2: Extracting reviews...")
chunks = []
reader = pd.read_json(DATA_PATH, lines=True, chunksize=CHUNK_SIZE,
                       dtype={"user_id": str, "parent_asin": str, "rating": float})

for i, chunk in enumerate(reader):
    filtered = chunk[chunk["user_id"].isin(sampled_users)]
    if len(filtered) > 0:
        chunks.append(filtered)
    print(f"  Chunk {i}: kept {len(filtered):,} / {len(chunk):,}")

df = pd.concat(chunks, ignore_index=True)
print(f"Final sample: {len(df):,} reviews from {df['user_id'].nunique():,} users")

# ── Save ────────────────────────────────────────────────────────
df.to_parquet("sample-pandas-with-chunk/sampled_reviews.parquet", index=False)
df.to_json("sample-pandas-with-chunk/sampled_reviews.jsonl", orient="records", lines=True)

Pass 1: Counting reviews per user...
  Chunk 0: 500,000 rows processed
  Chunk 1: 500,000 rows processed
  Chunk 2: 500,000 rows processed
  Chunk 3: 500,000 rows processed
  Chunk 4: 500,000 rows processed
  Chunk 5: 500,000 rows processed
  Chunk 6: 500,000 rows processed
  Chunk 7: 500,000 rows processed
  Chunk 8: 500,000 rows processed
  Chunk 9: 500,000 rows processed
  Chunk 10: 500,000 rows processed
  Chunk 11: 500,000 rows processed
  Chunk 12: 500,000 rows processed
  Chunk 13: 500,000 rows processed
  Chunk 14: 500,000 rows processed
  Chunk 15: 500,000 rows processed
  Chunk 16: 500,000 rows processed
  Chunk 17: 500,000 rows processed
  Chunk 18: 500,000 rows processed
  Chunk 19: 500,000 rows processed
  Chunk 20: 500,000 rows processed
  Chunk 21: 500,000 rows processed
  Chunk 22: 500,000 rows processed
  Chunk 23: 500,000 rows processed
  Chunk 24: 500,000 rows processed
  Chunk 25: 500,000 rows processed
  Chunk 26: 500,000 rows processed
  Chunk 27: 500,000 rows pro

### Third Sampling iteration Using Polars ###

Polars is a Rust-based DataFrame library that is 5-10x faster than pandas for this workload thanks to multi-threaded execution and lazy evaluation.

In [ ]:
import polars as pl
import random

DATA_PATH = "data/Books.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Polars lazy scan (memory-mapped, multi-threaded) ────────────
# This does NOT load the file into RAM — it creates a query plan
lf = pl.scan_ndjson(
    DATA_PATH,
    ignore_errors=True,
    low_memory=False,
    # Don't specify full schema — let Polars infer with overrides
    schema_overrides={
        "user_id": pl.Utf8,
        "rating": pl.Float64,
        "timestamp": pl.Int64,
    },
)

# ── Step 1: Find active users ──────────────────────────────────
print("Step 1: Identifying active users...")
user_review_counts = (
    lf.select("user_id")           # <── project FIRST, reduces data + schema issues
    .group_by("user_id")
    .agg(pl.len().alias("review_count"))
    .filter(pl.col("review_count") >= MIN_REVIEWS)
    .collect()  # materializes only the aggregation result
)
print(f"Active users (>= {MIN_REVIEWS}): {len(user_review_counts):,}")

# ── Step 2: Sample 50,000 users ────────────────────────────────
random.seed(SEED)
all_active = user_review_counts["user_id"].to_list()
sampled_users = random.sample(all_active, min(NUM_USERS, len(all_active)))
sampled_set = pl.Series("user_id", sampled_users)

# ── Step 3: Filter reviews ─────────────────────────────────────
print("Step 2: Filtering reviews for sampled users...")
sampled_df = (
    lf.filter(pl.col("user_id").is_in(sampled_set))
    .collect()
)
print(f"Sampled: {len(sampled_df):,} reviews, {sampled_df['user_id'].n_unique():,} users")

# ── Optional temporal filter (combine strategies) ───────────────
# Unix timestamp for Jan 1, 2020 = 1577836800
# Unix timestamp for Dec 31, 2023 = 1703980800
sampled_temporal = sampled_df.filter(
    (pl.col("timestamp") >= 1577836800) & (pl.col("timestamp") <= 1703980800)
)
print(f"After temporal filter (2020-2023): {len(sampled_temporal):,} reviews")

# ── Save ────────────────────────────────────────────────────────
sampled_df.write_parquet("sample-polars/sampled_reviews.parquet")
sampled_df.write_json("sample-polars/sampled_reviews.jsonl")

Step 1: Identifying active users...
Active users (>= 20): 137,305
Step 2: Filtering reviews for sampled users...


/tmp/ipykernel_3232/1172750435.py:44: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


Sampled: 2,427,273 reviews, 50,000 users
After temporal filter (2020-2023): 0 reviews


TypeError: DataFrame.write_json() got an unexpected keyword argument 'lines'

### Fourth Sampling iteration using Dask ###

Dask extends pandas to larger-than-memory datasets with lazy parallel execution.

In [ ]:
import dask.dataframe as dd
import random

DATA_PATH = "data/Books.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Load lazily (partitioned automatically) ─────────────────────
ddf = dd.read_json(
    DATA_PATH,
    lines=True,
    blocksize="256MB",  # each partition is ~256MB
    dtype={"user_id": str, "parent_asin": str, "rating": float},
    meta={"user_id": str, "parent_asin": str, "rating": float,
          "timestamp": int, "title": str, "text": str,
          "helpful_vote": int, "verified_purchase": bool},
)

# ── Step 1: Count reviews per user ─────────────────────────────
print("Counting reviews per user...")
user_counts = ddf.groupby("user_id").size().compute()  # returns pandas Series

active_users = user_counts[user_counts >= MIN_REVIEWS].index.tolist()
print(f"Active users: {len(active_users):,}")

# ── Step 2: Sample users ───────────────────────────────────────
random.seed(SEED)
sampled_users = set(random.sample(active_users, min(NUM_USERS, len(active_users))))

# ── Step 3: Filter ─────────────────────────────────────────────
print("Filtering reviews...")
sampled_ddf = ddf[ddf["user_id"].isin(sampled_users)]
sampled_df = sampled_ddf.compute()  # materialize to pandas

print(f"Sampled: {len(sampled_df):,} reviews")
sampled_df.to_parquet("sample-dask/sampled_reviews.parquet", index=False)
sampled_df.to_json("sample-dask/sampled_reviews.jsonl", lines=True)

Counting reviews per user...


### Fifth Sampling iteration using cuDF / RAPIDS ###

This is the GPU powerhouse. cuDF mirrors the pandas API but runs on NVIDIA GPUs. You need an NVIDIA GPU with sufficient VRAM (ideally 16GB+ for this dataset).

#### Option A ####

If the dataset fits in GPU VRAM (24GB+ GPU), cuDF can read JSONL directly on GPU.

In [ ]:
import cudf
import cupy as cp
import random

DATA_PATH = "data/Books.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Option A: If data fits in GPU VRAM (24GB+ GPU) ─────────────
# cuDF can read JSONL directly on GPU
gdf = cudf.read_json(DATA_PATH, lines=True, engine="cudf")

# Count reviews per user (GPU-accelerated groupby)
user_counts = gdf.groupby("user_id").size().reset_index(name="count")
active_users = user_counts[user_counts["count"] >= MIN_REVIEWS]["user_id"]
print(f"Active users: {len(active_users):,}")

# Sample users (on CPU — sampling is not GPU-bound)
random.seed(SEED)
active_list = active_users.to_pandas().tolist()
sampled = random.sample(active_list, min(NUM_USERS, len(active_list)))
sampled_series = cudf.Series(sampled)

# Filter on GPU
sampled_gdf = gdf[gdf["user_id"].isin(sampled_series)]
print(f"Sampled: {len(sampled_gdf):,} reviews")

# Convert to pandas or save
sampled_gdf.to_parquet("sample-cudf-rapids-24+/sampled_reviews.parquet")
sampled_gdf.to_json("sample-cudf-rapids-24+/sampled_reviews.jsonl", lines=True)

ModuleNotFoundError: No module named 'cudf'

#### Option B ####

Chunked GPU processing is more memory-efficient for limited VRAM.

In [ ]:
# ── Option B: Chunked GPU processing (limited VRAM) ────────────
import cudf
import random
from collections import Counter

DATA_PATH = "data/Books.jsonl"
CHUNK_SIZE = 2_000_000
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# Pass 1: Count on GPU in chunks
user_counts = Counter()
reader = cudf.read_json(DATA_PATH, lines=True, engine="cudf",
                         chunksize=CHUNK_SIZE)

for i, chunk in enumerate(reader):
    counts = chunk.groupby("user_id").size().reset_index(name="n")
    # Transfer small aggregation result to CPU
    pdf = counts.to_pandas()
    user_counts.update(dict(zip(pdf["user_id"], pdf["n"])))
    print(f"  GPU chunk {i} processed")

active_users = [u for u, c in user_counts.items() if c >= MIN_REVIEWS]
random.seed(SEED)
sampled_users = set(random.sample(active_users, min(NUM_USERS, len(active_users))))

# Pass 2: Filter on GPU in chunks
result_chunks = []
reader = cudf.read_json(DATA_PATH, lines=True, engine="cudf",
                         chunksize=CHUNK_SIZE)

for chunk in reader:
    filtered = chunk[chunk["user_id"].isin(list(sampled_users))]
    if len(filtered) > 0:
        result_chunks.append(filtered.to_pandas())

import pandas as pd
df = pd.concat(result_chunks, ignore_index=True)
df.to_parquet("sample-cudf-rapids-24-/sampled_reviews.parquet", index=False)
df.to_json("sample-cudf-rapids-24-/sampled_reviews.jsonl", lines=True)

#### Option C ####

In [ ]:
import cudf
import cupy as cp
import rmm


DATA_PATH = "data/Books.jsonl"
CHUNK_SIZE = 2_000_000
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# Configuration mémoire GPU
rmm.reinitialize(
    managed_memory=True,  # Unified memory CPU/GPU
    pool_allocator=True
)

def sample_with_gpu(DATA_PATH):
    """
    Échantillonnage GPU-accéléré avec cuDF (RAPIDS)
    
    Performance: 10-50x plus rapide que pandas
    Requis: GPU NVIDIA avec 8GB+ VRAM
    
    Installation:
    conda install -c rapidsai -c conda-forge -c nvidia \
        cudf cudatoolkit=13.1.1-1
    """
    
    # Lecture directe en GPU
    print("Chargement en GPU memory...")
    gdf = cudf.read_json(DATA_PATH, lines=True)
    
    # Optimisation types (GPU)
    gdf['rating'] = gdf['rating'].astype('int8')
    gdf['user_id'] = gdf['user_id'].astype('category')
    gdf['product_id'] = gdf['product_id'].astype('category')
    
    print(f"GPU memory utilisée: {gdf.memory_usage(deep=True).sum() / 1e9:.2f} GB")
    
    # Comptage ultra-rapide sur GPU
    user_counts = gdf['user_id'].value_counts()
    
    # Filtrer (opérations GPU)
    active_users = user_counts[user_counts >= 20].index
    
    # Échantillonnage
    selected_users = active_users[:50000]
    
    # Filtrage vectorisé GPU
    mask = gdf['user_id'].isin(selected_users)
    sample_gpu = gdf[mask]
    
    # Transfer vers CPU si nécessaire
    sample = sample_gpu.to_pandas()
        
    return sample

# Monitoring GPU
def monitor_gpu_memory():
    """Affiche utilisation GPU"""
    import pynvml
    
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    
    print(f"GPU Memory: {info.used/1e9:.2f}/{info.total/1e9:.2f} GB")
    pynvml.nvmlShutdown()

# Utilisation
if __name__ == '__main__':
    import time
    
    start = time.time()
    sample = sample_with_gpu(DATA_PATH)
    elapsed = time.time() - start
    
    print(f"\n⚡ Temps d'exécution GPU: {elapsed:.2f}s")
    print(f"   Reviews échantillonnées: {len(sample):,}")
    
    # Sauvegarde
    sample.to_parquet('sample-cudf-claude/sample_gpu_active_users.parquet', compression='snappy')
    sample.to_json("sample-cudf-claude/sample_gpu_active_users.jsonl", lines=True)

#### Option D ####

In [ ]:
import cudf
import cupy as cp
import rmm


DATA_PATH = "data/Books.jsonl"
CHUNK_SIZE = 2_000_000
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# Configuration mémoire GPU
rmm.reinitialize(
    managed_memory=True,  # Unified memory CPU/GPU
    pool_allocator=True
)

def sample_with_gpu(DATA_PATH):
    """
    Échantillonnage GPU-accéléré avec cuDF (RAPIDS)
    
    Performance: 10-50x plus rapide que pandas
    Requis: GPU NVIDIA avec 8GB+ VRAM
    
    Installation:
    conda install -c rapidsai -c conda-forge -c nvidia \
        cudf cudatoolkit=13.1.1-1
    """
    
    # Lecture directe en GPU
    print("Chargement en GPU memory...")
    gdf = cudf.read_json(DATA_PATH, lines=True)
    
    # Optimisation types (GPU)
    gdf['rating'] = gdf['rating'].astype('int8')
    gdf['user_id'] = gdf['user_id'].astype('category')
    gdf['product_id'] = gdf['product_id'].astype('category')
    
    print(f"GPU memory utilisée: {gdf.memory_usage(deep=True).sum() / 1e9:.2f} GB")
    
    # Conversion timestamp
    gdf['timestamp'] = cudf.to_datetime(gdf['timestamp'])
    gdf['year'] = gdf['timestamp'].dt.year
    
    # Échantillonnage stratifié par année (GPU)
    samples = []
    for year in gdf['year'].unique().to_pandas():
        year_data = gdf[gdf['year'] == year]
        n_samples = min(200000, len(year_data))
        samples.append(year_data.sample(n=n_samples))
    
    sample_gpu = cudf.concat(samples)
    sample = sample_gpu.to_pandas()

    return sample

# Monitoring GPU
def monitor_gpu_memory():
    """Affiche utilisation GPU"""
    import pynvml
    
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    
    print(f"GPU Memory: {info.used/1e9:.2f}/{info.total/1e9:.2f} GB")
    pynvml.nvmlShutdown()

# Utilisation
if __name__ == '__main__':
    import time
    
    start = time.time()
    sample = sample_with_gpu(DATA_PATH)
    elapsed = time.time() - start
    
    print(f"\n⚡ Temps d'exécution GPU: {elapsed:.2f}s")
    print(f"   Reviews échantillonnées: {len(sample):,}")
    
    # Sauvegarde
    sample.to_parquet('sample-cudf-claude/sample_gpu_temporal.parquet', compression='snappy')
    sample.to_json("sample-cudf-claude/sample_gpu_temporal.jsonl", lines=True)

### Sixth Sampling iteration using Dask with cuDF ###

Combines Dask's out-of-core scheduling with cuDF's GPU execution.


In [ ]:
import dask_cudf

DATA_PATH = "data/Books.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# Reads in partitions, each processed on GPU
ddf = dask_cudf.read_json(DATA_PATH, lines=True, blocksize="512MB")

# ── Step 1: Count reviews per user ─────────────────────────────
print("Counting reviews per user...")
user_counts = ddf.groupby("user_id").size().compute()  # returns pandas Series

active_users = user_counts[user_counts >= MIN_REVIEWS].index.tolist()
print(f"Active users: {len(active_users):,}")

# ── Step 2: Sample users ───────────────────────────────────────
random.seed(SEED)
sampled_users = set(random.sample(active_users, min(NUM_USERS, len(active_users))))

# ── Step 3: Filter ─────────────────────────────────────────────
print("Filtering reviews...")
sampled_ddf = ddf[ddf["user_id"].isin(sampled_users)]
sampled_df = sampled_ddf.compute()  # materialize to pandas

print(f"Sampled: {len(sampled_df):,} reviews")
sampled_df.to_parquet("sample-dask-cuda/sampled_reviews.parquet", index=False)
sampled_df.to_json("sample-dask-cuda/sampled_reviews.jsonl", lines=True)

### Seventh Sampling iteration using PySpark ###

For when you have a cluster or want Spark's optimizer.


In [ ]:
'''
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import random

spark = SparkSession.builder \
    .appName("AmazonReviewsSampling") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

DATA_PATH = "data/Books.jsonl"

# ── Load ────────────────────────────────────────────────────────
df = spark.read.json(DATA_PATH)

# ── Active users ────────────────────────────────────────────────
user_counts = df.groupBy("user_id").count().filter(F.col("count") >= 20)

# Sample 50K users using Spark's built-in sampling
# Approximate: calculate fraction needed
total_active = user_counts.count()
fraction = min(50000 / total_active, 1.0)
sampled_users = user_counts.sample(False, fraction, seed=42).limit(50000)

# ── Filter ──────────────────────────────────────────────────────
sampled_df = df.join(sampled_users.select("user_id"), on="user_id", how="inner")

# ── Optional temporal filter ────────────────────────────────────
sampled_df = sampled_df.filter(
    (F.col("timestamp") >= 1577836800) & (F.col("timestamp") <= 1703980800)
)

# ── Save ────────────────────────────────────────────────────────
sampled_df.coalesce(1).write.mode("overwrite").parquet("data/sampled_reviews_spark")

spark.stop()
'''

'\nfrom pyspark.sql import SparkSession\nfrom pyspark.sql import functions as F\nimport random\n\nspark = SparkSession.builder     .appName("AmazonReviewsSampling")     .config("spark.driver.memory", "8g")     .config("spark.sql.shuffle.partitions", "200")     .getOrCreate()\n\nDATA_PATH = "data/Books.jsonl"\n\n# ── Load ────────────────────────────────────────────────────────\ndf = spark.read.json(DATA_PATH)\n\n# ── Active users ────────────────────────────────────────────────\nuser_counts = df.groupBy("user_id").count().filter(F.col("count") >= 20)\n\n# Sample 50K users using Spark\'s built-in sampling\n# Approximate: calculate fraction needed\ntotal_active = user_counts.count()\nfraction = min(50000 / total_active, 1.0)\nsampled_users = user_counts.sample(False, fraction, seed=42).limit(50000)\n\n# ── Filter ──────────────────────────────────────────────────────\nsampled_df = df.join(sampled_users.select("user_id"), on="user_id", how="inner")\n\n# ── Optional temporal filter ───────

### Eighth Sampling iteration using DuckDB ###

DuckDB is an in-process OLAP database -- think "SQLite for analytics." Extremely fast for this kind of aggregation.

In [ ]:
import duckdb
import random

DATA_PATH = "data/Books.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

con = duckdb.connect()

# DuckDB reads JSONL natively and very efficiently
# ── Step 1: Find active users ──────────────────────────────────
print("Finding active users...")
active_users = con.execute(f"""
    SELECT user_id, COUNT(*) as cnt
    FROM read_json_auto('{DATA_PATH}', format='newline_delimited', maximum_object_size=10485760)
    GROUP BY user_id
    HAVING cnt >= {MIN_REVIEWS}
""").fetchdf()

print(f"Active users: {len(active_users):,}")

# ── Step 2: Sample users ───────────────────────────────────────
random.seed(SEED)
sampled = random.sample(active_users["user_id"].tolist(), 
                        min(NUM_USERS, len(active_users)))

# Register as a DuckDB table for efficient join
con.execute("CREATE TABLE sampled_users (user_id VARCHAR)")
con.executemany("INSERT INTO sampled_users VALUES (?)", [(u,) for u in sampled])

# ── Step 3: Extract reviews ────────────────────────────────────
print("Extracting reviews...")
con.execute(f"""
    COPY (
        SELECT r.*
        FROM read_json_auto('{DATA_PATH}', format='newline_delimited', 
                            maximum_object_size=10485760) r
        INNER JOIN sampled_users s ON r.user_id = s.user_id
    ) TO 'sample-duckdb/sampled_reviews.parquet' (FORMAT PARQUET)
""")

# ── With temporal filter ────────────────────────────────────────
con.execute(f"""
    COPY (
        SELECT r.*
        FROM read_json_auto('{DATA_PATH}', format='newline_delimited',
                            maximum_object_size=10485760) r
        INNER JOIN sampled_users s ON r.user_id = s.user_id
        WHERE r.timestamp >= 1577836800 AND r.timestamp <= 1703980800
    ) TO 'sample-duckdb/sampled_reviews_temporal.parquet' (FORMAT PARQUET)
""")

print("Done!")
con.close()